# Module 13: Intervention Analysis and Transfer Functions

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

Every intervention estimate so far has assumed a shape. A step says the effect
arrives all at once and stays. That is a modelling choice, and it is usually
made silently.

Intervention analysis, in the sense Box and Tiao gave the term, is about
estimating the **shape** of a response rather than assuming it: how fast it
arrives, whether it decays, whether it was a pulse rather than a step.

The honest finding of this module is that the shape is much harder to recover
than the size, and that the usual model selection tools will not tell you so.
Working out what a dataset cannot support is the last technical skill this
series teaches, and it is the one that stops a report from overreaching.

**About 35 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/OWNER/REPO/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
final = monthly[monthly["provisional"] == 0]          # never fit on unfinished months


CALENDAR = pd.period_range("2019-01", "2026-04", freq="M").to_timestamp()


def counts(agency_id):
    """Monthly counts on a complete calendar, so a gap stays visible as missing."""
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    s = pd.Series(d["n_uof"].values, dtype=float,
                  index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())
    s = s.reindex(CALENDAR)
    s.index.freq = "MS"
    return s


def rate(agency_id):
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    s = pd.Series((100 * d["n_uof"] / d["n_arrests"]).values,
                  index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())
    s = s.reindex(CALENDAR)
    s.index.freq = "MS"
    return s


print(f"{final['agency_id'].nunique()} agencies, {final['year_month'].nunique()} months")

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

TREATED = ["A001", "A002", "A004", "A007", "A010"]

f = final.copy()
f = f[~((f["agency_id"] == "A002") & (f["year_month"] == "2021-06"))]
f["treated"] = f["agency_id"].isin(TREATED).astype(int)
f["lo"] = np.log(f["n_arrests"])
pi = pd.PeriodIndex(f["year_month"], freq="M")
f["t"] = (pi.year.values - 2019) * 12 + pi.month.values - 1

START = (2023 - 2019) * 12 + 6        # 2023-07
f["k"] = f["t"] - START               # months since the programme started

pct = lambda b: 100 * (np.exp(b) - 1)


def poisson(formula, data):
    return smf.glm(formula, data, family=sm.families.Poisson(),
                   offset=data["lo"]).fit()


d = f[f["agency_id"] != "A007"].copy()
print(f"{d['agency_id'].nunique()} agencies, {len(d)} agency months")

## 2. The three classical shapes

| Shape | What it says | Where you see it |
|---|---|---|
| **Step** | the level changes and stays changed | a policy, a permanent staffing change |
| **Pulse** | one period is affected, then it returns | a single event, a weather month |
| **Gradual** | the effect builds toward a new level | training that spreads through a department |

The gradual case is a **first order transfer function**: each month the effect
moves a fixed fraction of the way from where it is to where it is heading. One
parameter, the decay, controls how fast.

The programme in this dataset is gradual by construction: 0, then 25, 58, 83
and 100 percent of a 12 percent reduction over five months.

## 3. The event study, and why it cannot see the shape

The assumption free approach is to estimate one coefficient per month relative
to the start and plot them.

In [ ]:
ev = d.copy()
ev["ek"] = np.clip(ev["k"], -6, 8).astype(int)
ev.loc[ev["treated"] == 0, "ek"] = -99          # controls never enter an event month

js = [j for j in range(-6, 9) if j != -1]       # month -1 is the reference
terms = " + ".join([f"I((treated==1)&(ek=={j}))" for j in js])
z = poisson("n_uof ~ C(agency_id) + C(year_month) + " + terms, ev)

truth = np.interp(js, [0, 1, 2, 3, 4], [0, .25, .58, .83, 1.0], left=0, right=1.0)
print("  month   estimate          95 percent interval      the truth")
for j, fr in zip(js, truth):
    key = [c for c in z.params.index if f"ek == {j}" in c][0]
    lo, hi = z.conf_int().loc[key]
    t_ = pct(np.log(0.88) * fr) if j >= 0 else 0.0
    print(f"   {j:+3d}   {pct(z.params[key]):+7.1f}%   "
          f"[{pct(lo):+7.1f}, {pct(hi):+7.1f}]      {t_:+6.1f}%")

The truth walks smoothly from 0 to −12 and stays. The estimates swing from
−34 to +24 percent, and the intervals run
from forty points wide at best to over eighty at worst.

**The event study is uninformative here, and it is the fashionable choice.** A
plot of these coefficients would show a jagged line that a reader will
interpret as dynamics. There are no dynamics in it. Four agencies and one
month of data per point is not enough to estimate anything month by month.

An event study is worth running to check the pre period coefficients, which
should be flat and near zero. As an estimate of a response shape at this sample
size it is not usable.

## 4. Three shapes, fitted properly

Instead of one coefficient per month, impose a shape and estimate its
amplitude. Three candidates, including the true one.

In [ ]:
d["step"] = ((d["treated"] == 1) & (d["k"] >= 4)).astype(float)
d["ramp"] = np.where(d["treated"] == 1, np.clip(d["k"] / 4.0, 0, 1), 0.0)
d["real"] = np.where(d["treated"] == 1,
                     np.interp(d["k"], [0, 1, 2, 3, 4], [0, .25, .58, .83, 1.0],
                               left=0, right=1.0), 0.0)

rows = []
for name, col in [("abrupt step at month 4", "step"),
                  ("linear ramp over 4 months", "ramp"),
                  ("the true phase in", "real")]:
    z = poisson(f"n_uof ~ C(agency_id) + C(year_month) + {col}", d)
    lo, hi = z.conf_int().loc[col]
    rows.append({"shape": name, "effect at full strength": f"{pct(z.params[col]):+.1f}%",
                 "interval": f"[{pct(lo):+.1f}, {pct(hi):+.1f}]",
                 "AIC": round(z.aic, 1)})
pd.DataFrame(rows).set_index("shape")

All three land within a point of the truth, and the AIC spread is 2.2 points.
**The wrong shape fits as well as the right one**, and the abrupt step, which
is the crudest of the three, has the best AIC and the closest estimate.

That is not bad luck. Over a 30 month settled window the three shapes differ
only in four months, so the data has almost nothing to distinguish them with.

## 5. Letting the data choose the decay

The transfer function version estimates the speed rather than assuming it.
There is no closed form, so grid search the decay parameter and let AIC pick.

In [ ]:
def transfer(data, delta):
    """A first order response: each month moves (1 - delta) of the way to full."""
    out = []
    for _, g in data.groupby("agency_id"):
        g = g.sort_values("t")
        x, vals = 0.0, []
        on = g["treated"].iloc[0] == 1
        for kk in g["k"].values:
            x = delta * x + (1 - delta) * (1.0 if (on and kk >= 0) else 0.0)
            vals.append(x)
        out.append(pd.Series(vals, index=g.index))
    return pd.concat(out).reindex(data.index)


settled = (d["treated"] == 1) & (d["k"] >= 4)
rows = []
for delta in [0.0, 0.3, 0.5, 0.7, 0.8, 0.9, 0.95, 0.97, 0.99]:
    d["tf"] = transfer(d, delta)
    z = poisson("n_uof ~ C(agency_id) + C(year_month) + tf", d)
    half = np.log(0.5) / np.log(delta) if 0 < delta < 1 else 0.0
    rows.append({"decay": delta, "AIC": round(z.aic, 1),
                 "half life, months": round(half, 1),
                 "effect at full strength": f"{pct(z.params['tf']):+.1f}%",
                 "average over settled months":
                     f"{pct(z.params['tf'] * d.loc[settled, 'tf'].mean()):+.1f}%"})
pd.DataFrame(rows).set_index("decay")

Read the AIC column, then read the two columns beside it.

**AIC improves all the way to the edge of the grid.** It never turns around.
Across the whole range it moves 2.4 points, which is nothing, while the
model's substantive claim moves from a 12 percent effect fully arrived within
a month to a **53 percent effect with a 69 month half life**.

The decay parameter is not identified. The likelihood surface is almost flat
and tilts very slightly toward the boundary, and AIC, asked to choose, walks
off the edge.

Now read the last column. **The average effect over the settled window sits
between 12 and 13 percent for every single value of the decay**, including the
absurd ones. The data pins down the average and says nothing about the path.

## 6. What is identified and what is not

| Quantity | Identified here | Why |
|---|---|---|
| Average effect over a stated window | **yes**, 12 to 13 percent | it is essentially a mean difference |
| Whether the effect is a step or a drift | no | [Module 11](Module_11_Interrupted_Time_Series.ipynb), the two trade off exactly |
| How fast the effect arrived | no | section 5, the decay runs to the boundary |
| Whether the effect is permanent | no | the series ends 30 months in |
| Month by month dynamics | no | section 3, intervals thirty points wide |

Only the first row belongs in a report as a number. The others belong in the
limitations paragraph, named explicitly, because a reader who is not told will
assume they were established.

**"We estimate an average reduction of 12.6 percent over the 30 months after
full implementation; the data cannot determine whether the effect arrived
abruptly or built over time"** is a stronger sentence than any claim about
dynamics, because it is one you can defend.

## 7. The handoff to causal inference

Everything in Part IV estimated an **association between a date and an
outcome**, as carefully as time series methods allow. What none of it did is
establish that the programme caused the change.

| The question this series can answer | The question it cannot |
|---|---|
| Did the rate change when the programme started | Would it have changed anyway |
| Is the change larger than the comparison agencies' | Are those agencies a valid counterfactual |
| Is it larger than noise | Did something else happen in November 2023 |
| Is the pre period consistent with the design | Why these five agencies adopted it |

That last row is the one that matters most here, and it is knowable:
[the answer key](../../../Data/GROUND_TRUTH.md) says the five agencies with the
**highest baseline rates** were selected. Selection on the outcome is the
oldest problem in evaluation, and no time series method touches it.

The [Causal Inference series](../../../Causal_Inference/) starts there.

## Exercise

The documented month of unrest at Tarnbridge, June 2021, is a pulse rather
than a step. Fit it as one and see what a pulse specification recovers.

In [ ]:
# Fill in the blank, then run.
SHAPE = None          # try "pulse", then "step", then "decay"

if SHAPE:
    g = f[f["agency_id"] == "A002"].sort_values("t").copy()
    full = final[final["agency_id"] == "A002"].sort_values("year_month")
    g = full.assign(
        lo=np.log(full["n_arrests"]),
        t=(pd.PeriodIndex(full["year_month"], freq="M").year - 2019) * 12
          + pd.PeriodIndex(full["year_month"], freq="M").month - 1)
    g["mo"] = pd.PeriodIndex(full["year_month"], freq="M").month
    g["yr"] = g["t"] / 12.0
    g["sin1"] = np.sin(2 * np.pi * g["mo"] / 12)
    g["cos1"] = np.cos(2 * np.pi * g["mo"] / 12)
    e = (2021 - 2019) * 12 + 5                      # 2021-06
    if SHAPE == "pulse":
        g["x"] = (g["t"] == e).astype(float)
    elif SHAPE == "step":
        g["x"] = (g["t"] >= e).astype(float)
    else:
        g["x"] = np.where(g["t"] >= e, 0.6 ** (g["t"] - e), 0.0)
    z = poisson("n_uof ~ yr + sin1 + cos1 + x", g)
    lo, hi = z.conf_int().loc["x"]
    print(f"  {SHAPE:6s}  effect {pct(z.params['x']):+8.1f}%  "
          f"[{pct(lo):+7.1f}, {pct(hi):+7.1f}]   AIC {z.aic:.1f}")
    print(f"          Pearson dispersion {z.pearson_chi2 / z.df_resid:.2f}")
else:
    print("Set SHAPE above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

Run all three.

```python
SHAPE = "pulse"    # then "step", then "decay"
```

The pulse wins decisively, and unlike section 5 the comparison is not close.
AIC is 575.9 for the pulse against 617.1 for the decaying version and 768.6
for the step, and the Pearson dispersion falls with it, from 4.73 to 1.41.

**Why does shape selection work here and fail in section 5?** Because the
signal is enormous and the shapes are genuinely different where the data is.
The unrest month runs about four and a half times the surrounding level, and
a pulse and a step disagree about 58 of the 88 months. The programme effect is 12 percent
and the three candidate shapes disagree about four months out of 88.

**Shape is identifiable when the effect is large relative to the noise and the
candidates differ over much of the sample.** Neither condition holds for a
gradual policy effect measured over a few years, which is the case people most
often want to fit a transfer function to.

Note what the pulse estimate is not. It is a description of one month, not an
estimate of what unrest does to use of force: n = 1. Fitting a shape is not the
same as learning a mechanism.

</details>

---

**Next:** [Module 14: Reporting, and Work That Outlives You](Module_14_Reporting_And_Reproducibility.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*